# COMP2050 - Attention U-Net for Medical Image Segmentation
## Notebook 4: Analysis & Report Figures

Load experiment results, run statistical tests, and generate all figures for the report.

In [ ]:
import sys
sys.path.insert(0, '.')

# Upload results.zip if starting fresh
# from google.colab import files
# uploaded = files.upload()  # Upload results.zip
# !unzip -o results.zip -d ./results

In [ ]:
# Load and display results
from evaluate import load_results, compute_statistics, generate_report_figures
import pandas as pd

df = load_results('./results')
print(f'Loaded {len(df)} experiment results')
df.head(20)

In [ ]:
# Statistical analysis
summary, pairwise = compute_statistics(df, metric='test_dice')

In [ ]:
# Show summary for all metrics
for metric in ['test_dice', 'test_iou', 'test_accuracy', 'test_sensitivity', 'test_specificity', 'test_hd95']:
    if metric in df.columns:
        print(f'\n--- {metric} ---')
        pivot = df.groupby(['architecture', 'loss'])[metric].agg(['mean', 'std'])
        print(pivot.to_string())

In [ ]:
# Generate all report figures
generate_report_figures(results_dir='./results', output_dir='./figures')
print('All figures saved to ./figures/')

In [ ]:
# Generate LaTeX table for the report
print('% Copy this into your LaTeX report:')
print('\\begin{table}[h]')
print('\\centering')
print('\\caption{Segmentation results (mean $\\pm$ std across 3 runs)}')
print('\\label{tab:results}')
print('\\begin{tabular}{llcccc}')
print('\\toprule')
print('Architecture & Loss & Dice & IoU & Sensitivity & HD95 \\\\\\midrule')

for _, row in df.groupby(['architecture', 'loss']).agg({
    'test_dice': ['mean', 'std'],
    'test_iou': ['mean', 'std'],
    'test_sensitivity': ['mean', 'std'],
    'test_hd95': ['mean', 'std'],
}).reset_index().iterrows():
    arch = row['architecture']
    loss = row['loss']
    dice = f"{row[('test_dice', 'mean')]:.3f} \\pm {row[('test_dice', 'std')]:.3f}"
    iou = f"{row[('test_iou', 'mean')]:.3f} \\pm {row[('test_iou', 'std')]:.3f}"
    sens = f"{row[('test_sensitivity', 'mean')]:.3f} \\pm {row[('test_sensitivity', 'std')]:.3f}"
    hd = f"{row[('test_hd95', 'mean')]:.1f} \\pm {row[('test_hd95', 'std')]:.1f}"
    print(f'{arch} & {loss} & {dice} & {iou} & {sens} & {hd} \\\\\\\n')

print('\\bottomrule')
print('\\end{tabular}')
print('\\end{table}')

In [ ]:
# Download all figures
import shutil
shutil.make_archive('figures', 'zip', './figures')
from google.colab import files
files.download('figures.zip')